In [ ]:
from pathlib import Path
import csv
import random
import shutil
import urllib.request
import zipfile

ROOT = Path("/content/TrashNet")
ZIP_FILE = ROOT / "dataset-resized.zip"
SOURCE = ROOT / "dataset-resized"
OUTPUT = ROOT / "dataset-split"
SEED = 42
URL = "https://raw.githubusercontent.com/garythung/trashnet/master/data/dataset-resized.zip"

EXCLUDED = {
    "glass176.jpg", "plastic152.jpg",
    "glass115.jpg", "metal91.jpg",
    "glass389.jpg", "plastic332.jpg",
}
!pip install -q codecarbon

### 1. Environment, paths and reproducibility

**Purpose:** Imports the required Python tools, defines the Colab storage paths, and fixes the random seed at 42 so every group member creates the same split. The six filenames in `EXCLUDED` are exact duplicate images with conflicting class labels, so they are excluded from model development.

In [ ]:
ROOT.mkdir(exist_ok=True)

if not ZIP_FILE.exists():
    print("Downloading TrashNet from the official repository...")
    urllib.request.urlretrieve(URL, ZIP_FILE)
else:
    print("ZIP file already exists; download skipped.")

if not SOURCE.exists():
    print("Extracting TrashNet...")
    with zipfile.ZipFile(ZIP_FILE) as archive:
        archive.extractall(ROOT)
else:
    print("Dataset already extracted; extraction skipped.")

print("Dataset source:", SOURCE)

### 2. Download and extract the dataset

**Purpose:** Downloads the resized TrashNet ZIP directly from the official repository and extracts it inside the Colab runtime. The existence checks make this cell safe to run more than once during the same session.

In [ ]:
if OUTPUT.exists():
    print("dataset-split already exists; keeping the existing split.")
else:
    rng = random.Random(SEED)
    manifest = []

    for class_dir in sorted(path for path in SOURCE.iterdir() if path.is_dir()):
        images = sorted(
            path for path in class_dir.glob("*.jpg")
            if path.name not in EXCLUDED
        )
        rng.shuffle(images)

        train_end = round(len(images) * 0.70)
        validation_end = train_end + round(len(images) * 0.15)
        groups = {
            "train": images[:train_end],
            "validation": images[train_end:validation_end],
            "test": images[validation_end:],
        }

        for split_name, split_images in groups.items():
            destination = OUTPUT / split_name / class_dir.name
            destination.mkdir(parents=True, exist_ok=True)
            for image in split_images:
                shutil.copy2(image, destination / image.name)
                manifest.append([image.name, class_dir.name, split_name])

    with (OUTPUT / "split_manifest.csv").open("w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["filename", "label", "split"])
        writer.writerows(manifest)

    print("Dataset split created:", OUTPUT)

### 3. Create the train, validation and test splits

**Purpose:** Randomly shuffles each class separately and creates a stratified 70% training, 15% validation and 15% test split. Images are copied into class folders, while `split_manifest.csv` records the assigned label and split. If the split already exists, the code keeps it instead of raising an error or changing the test set.

In [ ]:
expected = {"train": 1765, "validation": 378, "test": 378}
actual = {
    split: len(list((OUTPUT / split).glob("*/*.jpg")))
    for split in expected
}

assert actual == expected, f"Unexpected split counts: {actual}"
print("Split verification passed:", actual)
print("Total usable images:", sum(actual.values()))

### 4. Verify the completed split

**Purpose:** Counts the images in each folder and stops immediately if the result differs from the planned split. A successful output must be `train: 1765`, `validation: 378`, and `test: 378`.

---

## Week 9: Frozen-backbone MobileNetV3 baseline

The following cells load the prepared data, freeze an ImageNet-pretrained MobileNetV3 backbone, train only the new classification head, and record the baseline performance. Run them from top to bottom after the four data-preparation sections.

In [ ]:
import json
import time
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
from sklearn.utils.class_weight import compute_class_weight

tf.keras.utils.set_random_seed(SEED)
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    OUTPUT / "train", image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, shuffle=True, seed=SEED
)
class_names = train_ds.class_names
validation_ds = tf.keras.utils.image_dataset_from_directory(
    OUTPUT / "validation", image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, shuffle=False, class_names=class_names
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    OUTPUT / "test", image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, shuffle=False, class_names=class_names
)

y_train_all = np.concatenate([y.numpy() for _, y in train_ds])
raw_weights = compute_class_weight("balanced", classes=np.unique(y_train_all), y=y_train_all)

smooth_weights = np.sqrt(raw_weights)
class_weights = dict(enumerate(smooth_weights))

train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
validation_ds = validation_ds.cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.cache().prefetch(tf.data.AUTOTUNE)

print("Class order:", class_names)
print("Smoothed class weights:", {class_names[k]: round(v, 3) for k, v in class_weights.items()})

### 5. Load the prepared image datasets

**Purpose:** Reads the three fixed split folders, resizes every image to 224 × 224, uses batches of 32, and preserves the same alphabetical class order across training, validation and test data. Only the training loader shuffles images.

In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.15),
])

backbone = tf.keras.applications.MobileNetV3Small(
    weights="imagenet",
    include_top=False,
    input_shape=IMG_SIZE + (3,),
)
backbone.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = augmentation(inputs)
x = backbone(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation="hard_swish")(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

assert backbone.trainable is False
print("Frozen backbone confirmed:", not backbone.trainable)
model.summary()

### 6. Build the frozen-backbone baseline

**Purpose:** Apply training-only augmentation and load ImageNet-pretrained MobileNetV3Small. Freeze all backbone parameters and train only the new six-class Dense head. Global pooling and dropout have no trainable parameters. The confirmation must read `Frozen backbone confirmed: True`.

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
)
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-5,
    verbose=1,
)

history = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=[early_stopping, lr_scheduler],
)

best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
print("Best epoch:", best_epoch)
print("Best validation accuracy:", max(history.history["val_accuracy"]))

### 7. Train only the classification head

**Purpose:** Trains the unfrozen output layers for at most 15 epochs. Early stopping monitors validation loss, stops after three non-improving epochs, and restores the best weights. The test set is not used during training.

In [ ]:
start = time.perf_counter()
probabilities = model.predict(test_ds, verbose=0)
elapsed = time.perf_counter() - start

y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
y_pred = probabilities.argmax(axis=1)
confidence = probabilities.max(axis=1)

report = classification_report(
    y_true, y_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)

results = {
    "test_accuracy": float(np.mean(y_true == y_pred)),
    "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
    "manual_review_rate_below_0.60": float(np.mean(confidence < 0.60)),
    "milliseconds_per_image": float(elapsed / len(y_true) * 1000),
    "trash_precision_proxy": report["trash"]["precision"],
    "trash_recall_proxy": report["trash"]["recall"],
}

print(json.dumps(results, indent=2))
print(classification_report(
    y_true, y_pred,
    target_names=class_names,
    zero_division=0,
))

### 8. Evaluate the untouched test set

**Purpose:** Makes one final prediction pass over the test set and records test accuracy, macro-F1, per-class precision/recall/F1, processing time, and the proportion of predictions below 0.60 confidence. The `trash` metrics are only a provisional contamination proxy, not a complete City of Sydney rules assessment.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="validation")
axes[0].set_title("Accuracy by epoch")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="validation")
axes[1].set_title("Loss by epoch")
axes[1].set_xlabel("Epoch")
axes[1].legend()

ConfusionMatrixDisplay(
    confusion_matrix(y_true, y_pred),
    display_labels=class_names,
).plot(ax=axes[2], xticks_rotation=45, cmap="Blues", colorbar=False)
axes[2].set_title("Test confusion matrix")
plt.tight_layout()
plt.show()

MODEL_PATH = ROOT / "frozen_mobilenetv3_baseline.keras"
RESULTS_PATH = ROOT / "baseline_results.json"
model.save(MODEL_PATH)
with RESULTS_PATH.open("w") as file:
    json.dump(results, file, indent=2)

print("Model saved to:", MODEL_PATH)
print("Results saved to:", RESULTS_PATH)

### 9. Visualise and save the baseline evidence

**Purpose:** Plots training/validation accuracy and loss to reveal overfitting, displays the test confusion matrix to show class-specific errors, and saves both the trained baseline model and the numerical results. These outputs provide the evidence needed for the Week 9 progress record and the later comparison with the Week 10 fine-tuned model.

**Run order:** Execute code sections 1–9 from top to bottom. Section 7 starts model training; sections 8–9 must be run only after training finishes.

## Week 10: Fine-tune MobileNetV3 and compare

**Updated project scope:** EcoSort is a software prototype that accepts an uploaded image or a camera photo of one waste item, predicts one of six material categories, and displays confidence and a review prompt. Hardware integration, municipal audit automation and LLM summaries are outside the revised scope. Material classification does not establish local bin eligibility.

### 10. Load a separate model for fine-tuning

Load the trained Week 9 checkpoint without modifying the frozen baseline. Verify the split manifest and check exact image hashes for cross-split duplicates. After a Colab runtime reset, restore sections 1–9 before running this section.

In [ ]:
import hashlib
import pandas as pd

BASELINE_PATH = ROOT / "frozen_mobilenetv3_baseline.keras"
assert BASELINE_PATH.exists(), "Run sections 1-9 to restore the baseline first."
with (ROOT / "baseline_results.json").open() as file:
    baseline_results = json.load(file)

split_manifest = pd.read_csv(OUTPUT / "split_manifest.csv")
assert split_manifest.groupby("split").size().to_dict() == {"test": 378, "train": 1765, "validation": 378}
hash_sets = {}
for split in ("train", "validation", "test"):
    hash_sets[split] = {
        hashlib.sha256(path.read_bytes()).hexdigest()
        for path in (OUTPUT / split).glob("*/*.jpg")
    }
assert not (hash_sets["train"] & hash_sets["test"])
assert not (hash_sets["train"] & hash_sets["validation"])
assert not (hash_sets["validation"] & hash_sets["test"])
manifest_sha256 = hashlib.sha256((OUTPUT / "split_manifest.csv").read_bytes()).hexdigest()

fine_model = tf.keras.models.load_model(BASELINE_PATH, compile=False)
fine_backbone = next(layer for layer in fine_model.layers if isinstance(layer, tf.keras.Model) and "mobilenet" in layer.name.lower())
print("Baseline loaded; no cross-split exact duplicates.")
print("Manifest SHA256:", manifest_sha256)
print("Current baseline:", baseline_results)

### 11. Unfreeze the final layers

Unfreeze trainable layers within the last 30 backbone layers while keeping Batch Normalization frozen. Recompile with Adam at a learning rate of 1e-5. Preserve the baseline preprocessing and training-only augmentation.

In [ ]:
tf.keras.utils.set_random_seed(SEED)
fine_backbone.trainable = True
for index, layer in enumerate(fine_backbone.layers):
    layer.trainable = (
        index >= len(fine_backbone.layers) - 30
        and not isinstance(layer, tf.keras.layers.BatchNormalization)
    )
fine_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5), # 从 1e-5 提升至 3e-5，加速 CPU 下的收敛
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
assert fine_backbone.trainable_weights
assert all(not layer.trainable for layer in fine_backbone.layers if isinstance(layer, tf.keras.layers.BatchNormalization))
print("Trainable parameters:", sum(int(np.prod(w.shape)) for w in fine_model.trainable_weights))
print("Trainable backbone layers:", [layer.name for layer in fine_backbone.layers if layer.trainable_weights])

### 12. Fine-tune using training and validation data

Train for at most 10 additional epochs. Select the checkpoint by validation loss, stop after three non-improving epochs, and restore the best checkpoint. The test set is excluded from training and checkpoint selection. Save a CSV training log and a separate fine-tuned model.

In [ ]:
import pandas as pd
from codecarbon import EmissionsTracker

W10 = ROOT / "week10"
W10.mkdir(exist_ok=True)
FT_PATH = W10 / "finetuned_mobilenetv3.keras"

# 1. Initialize the EmissionsTracker and output the emissions.csv to the week10 directory
tracker = EmissionsTracker(
    project_name="EcoSort_MobileNetV3_Finetuning",
    output_dir=str(W10),
    log_level="warning",          # Filter out verbose polling logs
    measure_power_secs=5,         # Poll hardware power every 5 seconds
)

fine_lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1,
)

print("Starting carbon emissions tracker...")
# Start tracking power consumption before training
tracker.start()
training_start = time.perf_counter()

fine_history = fine_model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(str(FT_PATH), monitor="val_loss", save_best_only=True),
        tf.keras.callbacks.CSVLogger(str(W10 / "fine_training_log.csv")),
        fine_lr_scheduler,
    ],
)

fine_training_seconds = time.perf_counter() - training_start

# Stop tracking and retrieve the estimated carbon emissions (kg CO2e)
fine_emissions_kg = tracker.stop()

# 2. Read the generated CSV to extract the exact energy consumption (kWh)
emissions_df = pd.read_csv(W10 / "emissions.csv")
latest_run = emissions_df.iloc[-1]
fine_energy_kwh = float(latest_run["energy_consumed"])

fine_model = tf.keras.models.load_model(FT_PATH, compile=False)
best_fine_epoch = int(np.argmin(fine_history.history["val_loss"]) + 1)

print("-" * 50)
print("Best fine-tuning epoch:", best_fine_epoch)
print("Validation accuracy at selected epoch:", fine_history.history["val_accuracy"][best_fine_epoch - 1])
print(f"Training duration: {fine_training_seconds:.2f} seconds")
print(f"Energy consumed: {fine_energy_kwh:.6f} kWh")
print(f"Operational Carbon emissions: {fine_emissions_kg:.6f} kg CO2e")
print("-" * 50)

### 13. Compare both models on the identical test set

Evaluate the frozen baseline and the fine-tuned model using the same split and metrics: accuracy, macro-F1, per-class precision/recall/F1 and the review rate at the fixed 0.60 threshold. Warm up inference before timing; timing includes image loading and batch inference, not deployment latency. Save predictions, reports, confusion matrices, curves and metric differences. The test set was inspected in Week 9, so this is a course comparison rather than a new blind evaluation. Independent real-photo testing remains necessary.

In [ ]:
labels = np.arange(len(class_names))
true_labels = np.concatenate([y.numpy() for _, y in test_ds])
assert len(true_labels) == 378
file_paths = sorted((OUTPUT / "test").glob("*/*.jpg"))
comparison_rows = []

for name, candidate in [("frozen_baseline", tf.keras.models.load_model(BASELINE_PATH, compile=False)), ("fine_tuned", fine_model)]:
    candidate.predict(test_ds.take(1), verbose=0)
    started = time.perf_counter()
    probs = candidate.predict(test_ds, verbose=0)
    milliseconds = (time.perf_counter() - started) * 1000 / len(true_labels)
    assert probs.shape == (378, 6) and np.isfinite(probs).all()
    predicted = probs.argmax(axis=1)
    detailed = classification_report(true_labels, predicted, labels=labels, target_names=class_names, output_dict=True, zero_division=0)

    row = {
        "model": name,
        "accuracy": float(np.mean(predicted == true_labels)),
        "macro_f1": float(f1_score(true_labels, predicted, labels=labels, average="macro")),
        "trash_precision_proxy": detailed["trash"]["precision"],
        "trash_recall_proxy": detailed["trash"]["recall"],
        "manual_review_rate_0.60": float(np.mean(probs.max(axis=1) < 0.60)),
        "milliseconds_per_image": milliseconds,
    }
    comparison_rows.append(row)
    pd.DataFrame(detailed).transpose().to_csv(W10 / f"{name}_classification_report.csv")
    pd.DataFrame({
        "filename": [str(p.relative_to(OUTPUT / "test")) for p in file_paths],
        "true_label": [class_names[i] for i in true_labels],
        "predicted_label": [class_names[i] for i in predicted],
        "confidence": probs.max(axis=1),
    }).to_csv(W10 / f"{name}_predictions.csv", index=False)

    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay(confusion_matrix(true_labels, predicted, labels=labels), display_labels=class_names).plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
    ax.set_title(name)
    fig.tight_layout()
    fig.savefig(W10 / f"{name}_confusion_matrix.png", dpi=180)
    plt.show()

comparison = pd.DataFrame(comparison_rows).set_index("model")
comparison.to_csv(W10 / "baseline_vs_finetuned.csv")
display(comparison)
print("Fine-tuned minus baseline:")
print(comparison.loc["fine_tuned"] - comparison.loc["frozen_baseline"])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric in zip(axes, ["accuracy", "loss"]):
    ax.plot(fine_history.history[metric], label="train")
    ax.plot(fine_history.history["val_" + metric], label="validation")
    ax.set(title="Fine-tuning " + metric, xlabel="Epoch (zero-based)")
    ax.legend()
fig.tight_layout()
fig.savefig(W10 / "fine_training_curves.png", dpi=180)
plt.show()

# Inject the measured carbon footprint data into the final reporting metadata dictionary
metadata = {
    "split_manifest_sha256": manifest_sha256,
    "seed": SEED,
    "image_size": list(IMG_SIZE),
    "batch_size": BATCH_SIZE,
    "learning_rate": 3e-5,
    "unfreeze_last_layers": 30,
    "batch_normalization_frozen": True,
    "best_epoch": best_fine_epoch,
    "training_seconds": fine_training_seconds,
    # --- Green AI & Sustainability Audit Metrics ---
    "sustainability_metrics": {
        "energy_consumed_kwh": fine_energy_kwh,
        "emissions_kg_co2e": fine_emissions_kg,
        "hardware_scope": "CPU/GPU Hardware Telemetry via CodeCarbon",
        "boundary_note": "Operational energy during fine-tuning (excluding embodied carbon/PUE overhead)"
    },
    # -----------------------------------------------
    "tensorflow_version": tf.__version__,
    "class_names": class_names,
    "metrics": comparison_rows,
    "note": "Baseline reconstructed after runtime reset; compare paired models from this run. Trash is a contamination proxy. Test has previously been inspected.",
}

(W10 / "week10_results.json").write_text(json.dumps(metadata, indent=2))
shutil.copy2(OUTPUT / "split_manifest.csv", W10 / "split_manifest.csv")
shutil.copy2(BASELINE_PATH, W10 / BASELINE_PATH.name)
assert FT_PATH.exists() and (W10 / "baseline_vs_finetuned.csv").exists()
print("WEEK 10 COMPLETE. Results (including emissions.csv):", W10)

### 14. Preserve experiment evidence

The comparison section saves both models, the split manifest, training logs, predictions and figures under `/content/TrashNet/week10`. Section 18 downloads this evidence as a ZIP. Keep the archive because Colab runtime storage is temporary.

## EcoSort software prototype

### 15. Install the image interface

Install Gradio only when unavailable. The prototype accepts an upload or webcam snapshot. Camera access is requested when the user chooses to take a photo. The classifier covers six material categories; it does not determine local bin eligibility. Use one item per photo.

In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("gradio") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio"])
import gradio as gr
from PIL import Image, ImageOps
print("Gradio:", gr.__version__)

### 16. Prepare and check image classification

Select the app model by validation loss. Convert an image to RGB, correct orientation and apply the same resizing as training. Return all six scores and flag confidence below 0.60. Softmax scores are not calibrated probabilities of correctness. A self-check verifies finite scores that sum to one.

In [ ]:
use_finetuned = min(fine_history.history["val_loss"]) < min(history.history["val_loss"])
app_model = fine_model if use_finetuned else tf.keras.models.load_model(BASELINE_PATH, compile=False)
app_model_name = "Fine-tuned MobileNetV3" if use_finetuned else "Frozen MobileNetV3 baseline"
DISPLAY_NAMES = ["Cardboard", "Glass", "Metal", "Paper", "Plastic", "Other waste (TrashNet trash)"]

def classify_waste(image):
    if image is None:
        raise gr.Error("Upload an image or take a photo first.")
    image = ImageOps.exif_transpose(image).convert("RGB")
    pixels = tf.image.resize(np.asarray(image, dtype=np.float32), IMG_SIZE, method="bilinear")
    scores = app_model(np.expand_dims(pixels.numpy(), 0), training=False).numpy()[0]
    if scores.shape != (6,) or not np.isfinite(scores).all():
        raise gr.Error("The model could not process this image. Try another photo.")
    index = int(scores.argmax())
    confidence = float(scores[index])
    status = "Manual review recommended" if confidence < 0.60 else "Prediction available"
    return dict(zip(DISPLAY_NAMES, map(float, scores))), f"{status} | {DISPLAY_NAMES[index]} | confidence: {confidence:.1%}"

sample_path = next((OUTPUT / "test").glob("*/*.jpg"))
with Image.open(sample_path) as sample:
    checked_scores, checked_message = classify_waste(sample)
assert len(checked_scores) == 6 and abs(sum(checked_scores.values()) - 1.0) < 1e-4
print("Image pipeline check passed.")
print("App model selected by validation loss:", app_model_name)

### 17. Launch EcoSort

Upload a photo or select the webcam input, then run classification. The temporary Gradio interface remains available while this Colab runtime is active. One item per image is expected.

In [ ]:
if "demo" in globals():
    demo.close()
example_images = [[str(next((OUTPUT / "test" / label).glob("*.jpg"))) ] for label in class_names]
demo = gr.Interface(
    fn=classify_waste,
    inputs=gr.Image(sources=["upload", "webcam"], type="pil", label="Waste item"),
    outputs=[gr.Label(num_top_classes=6, label="Material category"), gr.Textbox(label="Review status", interactive=False)],
    examples=example_images,
    title="EcoSort AI",
    flagging_mode="never",
)
demo.launch(share=True, inline=True, debug=False)

### 18. Preserve the software and experiment results

Download a bundle containing the trained models and Week 10 evidence. The source code remains in this Notebook. Keep this archive to avoid losing checkpoints when the runtime resets.

In [ ]:
from google.colab import files
for manifest_name in ("combined_inventory.csv", "exact_duplicates.csv", "split_manifest_v2.csv"):
    manifest_path = Path("/content/EcoSort_dataset_v2") / manifest_name
    if manifest_path.exists():
        shutil.copy2(manifest_path, W10 / manifest_name)
(W10 / "app_config.json").write_text(json.dumps({"model": app_model_name, "classes": class_names, "review_threshold": 0.60}, indent=2))
bundle = shutil.make_archive(str(ROOT / "EcoSort_Week10_results"), "zip", W10)
print("Evidence bundle:", bundle)
files.download(bundle)

## Expanded dataset v2: TrashNet + RealWaste

### 19. Download RealWaste and audit the combined inventory

Download RealWaste from UCI (Single, Iranmanesh and Raad, 2023; https://doi.org/10.24432/C5SS4G, CC BY 4.0). Preserve nine material classes, mapping Miscellaneous Trash to trash and retaining food_organics, textile_trash and vegetation as separate classes. Combine the inventory with 2,521 retained TrashNet images and check image decoding and exact SHA256 duplicates. This does not alter the fixed six-class Week 9/10 experiment. Training on the expanded taxonomy requires a separate split and nine-class model.

In [ ]:
from pathlib import Path
import hashlib
import urllib.request
import zipfile
from PIL import Image

REAL_ROOT = Path("/content/RealWaste")
REAL_ROOT.mkdir(exist_ok=True)
REAL_ZIP = REAL_ROOT / "realwaste.zip"
if not REAL_ZIP.exists():
    print("Downloading RealWaste from UCI...")
    urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/908/realwaste.zip", REAL_ZIP)
REAL_SOURCE = REAL_ROOT / "realwaste-main" / "RealWaste"
if not REAL_SOURCE.exists():
    with zipfile.ZipFile(REAL_ZIP) as archive:
        archive.extractall(REAL_ROOT)
assert REAL_SOURCE.exists(), "Unexpected RealWaste archive layout."
class_mapping = {
    "Cardboard": "cardboard", "Food Organics": "food_organics",
    "Glass": "glass", "Metal": "metal", "Miscellaneous Trash": "trash",
    "Paper": "paper", "Plastic": "plastic", "Textile Trash": "textile_trash",
    "Vegetation": "vegetation",
}
real_images = sorted(p for p in REAL_SOURCE.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
assert len(real_images) == 4752, f"Unexpected RealWaste count: {len(real_images)}"
trashnet_images = sorted(p for p in SOURCE.glob("*/*.jpg") if p.name not in EXCLUDED)
rows = []
for dataset, paths in [("TrashNet", trashnet_images), ("RealWaste", real_images)]:
    for path in paths:
        label = path.parent.name if dataset == "TrashNet" else class_mapping[path.parent.name]
        try:
            with Image.open(path) as image:
                image.verify()
            status = "valid"
        except Exception as error:
            status = "invalid: " + str(error)
        rows.append({"dataset": dataset, "label": label, "path": str(path), "sha256": hashlib.sha256(path.read_bytes()).hexdigest(), "status": status})
inventory = pd.DataFrame(rows)
V2 = Path("/content/EcoSort_dataset_v2")
V2.mkdir(exist_ok=True)
inventory.to_csv(V2 / "combined_inventory.csv", index=False)
duplicates = inventory[inventory.duplicated("sha256", keep=False)]
duplicates.to_csv(V2 / "exact_duplicates.csv", index=False)
valid_inventory = inventory[inventory.status == "valid"]
unique_inventory = valid_inventory.drop_duplicates("sha256")
print("RealWaste images:", len(real_images))
print("TrashNet retained images:", len(trashnet_images))
print("Combined images before new deduplication:", len(inventory))
print("Valid unique images:", len(unique_inventory))
print("Images still needed for 10,000:", max(0, 10000 - len(unique_inventory)))
print("Invalid images:", len(inventory) - len(valid_inventory))
display(pd.crosstab(inventory.label, inventory.dataset, margins=True))
assert len(inventory) == 7273
print("Dataset v2 inventory saved:", V2)

### 20. Create a separate nine-class dataset version

Remove invalid images, exact duplicates and all images with conflicting labels for the same hash. Stratify by source dataset and material class, then copy images into a separate seed-42 70/15/15 split. Source-prefixed filenames prevent collisions. This prepares expanded data without changing the six-class baseline comparison. The current trained models and interface remain six-class until a nine-class model is trained.

In [ ]:
conflicting_hashes = set(valid_inventory.groupby("sha256").label.nunique().loc[lambda x: x > 1].index)
clean_v2 = valid_inventory[~valid_inventory.sha256.isin(conflicting_hashes)].drop_duplicates("sha256").copy()
V2_SPLIT = V2 / "dataset-split"
rng_v2 = random.Random(42)
v2_rows = []
for (dataset_name, label), group in clean_v2.groupby(["dataset", "label"], sort=True):
    records = group.sort_values("path").to_dict("records")
    rng_v2.shuffle(records)
    train_end = round(len(records) * 0.70)
    val_end = train_end + round(len(records) * 0.15)
    for split, subset in [("train", records[:train_end]), ("validation", records[train_end:val_end]), ("test", records[val_end:])]:
        for record in subset:
            destination = V2_SPLIT / split / label / (dataset_name + "_" + Path(record["path"]).name)
            destination.parent.mkdir(parents=True, exist_ok=True)
            if not destination.exists():
                shutil.copy2(record["path"], destination)
            v2_rows.append({**record, "split": split, "split_path": str(destination)})
v2_manifest = pd.DataFrame(v2_rows)
assert v2_manifest.sha256.is_unique
assert len(v2_manifest) == len(clean_v2)
assert v2_manifest.label.nunique() == 9
assert v2_manifest.groupby("sha256").split.nunique().max() == 1
v2_manifest.to_csv(V2 / "split_manifest_v2.csv", index=False)
print("Prepared nine-class images:", len(v2_manifest))
print("Remaining to reach 10,000:", max(0, 10000 - len(v2_manifest)))
print("Conflicting-label hashes excluded:", len(conflicting_hashes))
display(pd.crosstab(v2_manifest.label, v2_manifest.split, margins=True))
print("Expanded split saved to:", V2_SPLIT)

## Dataset v3: 12 categories and balanced training

### 21. Download additional research datasets

Add E-Waste Image Dataset (Kaggle, akshat103; Apache 2.0), Household Garbage (mostafaabla; database ODbL, image rights retained by original authors; research use), and Waste Classification (PhenomSG; MIT as declared by its uploader). Retain source metadata and do not redistribute third-party raw images in the results bundle. Dataset-level licences do not verify rights for every web-scraped image. Inspect content before inclusion.

In [ ]:
import concurrent.futures
import json
import urllib.request
import zipfile
from pathlib import Path

V3 = Path("/content/EcoSort_dataset_v3")
RAW3 = V3 / "raw"
RAW3.mkdir(parents=True, exist_ok=True)
SOURCE_REFS = {
    "household": "mostafaabla/garbage-classification",
    "ewaste": "akshat103/e-waste-image-dataset",
    "phenom": "phenomsg/waste-classification",
}
def download_research_dataset(item):
    name, ref = item
    metadata_url = "https://www.kaggle.com/api/v1/datasets/view/" + ref
    with urllib.request.urlopen(metadata_url, timeout=60) as response:
        metadata = json.load(response)
    (RAW3 / (name + "_metadata.json")).write_text(json.dumps(metadata, indent=2))
    archive_path = RAW3 / (name + ".zip")
    if not archive_path.exists():
        partial = archive_path.with_suffix(".partial")
        urllib.request.urlretrieve("https://www.kaggle.com/api/v1/datasets/download/" + ref, partial)
        partial.rename(archive_path)
    destination = RAW3 / name
    marker = destination / ".extracted"
    if not marker.exists():
        destination.mkdir(exist_ok=True)
        with zipfile.ZipFile(archive_path) as archive:
            for member in archive.infolist():
                target = (destination / member.filename).resolve()
                if not target.is_relative_to(destination.resolve()):
                    raise ValueError("Unsafe archive member: " + member.filename)
            archive.extractall(destination)
        marker.touch()
    return name, len(list(destination.rglob("*")))

with concurrent.futures.ThreadPoolExecutor(max_workers=3) as pool:
    for result in pool.map(download_research_dataset, SOURCE_REFS.items()):
        print("Downloaded:", result)
print("Raw image folders:")
for name in SOURCE_REFS:
    counts = {}
    for path in (RAW3 / name).rglob("*"):
        if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}:
            counts[path.parent.name] = counts.get(path.parent.name, 0) + 1
    print(name, counts)

### 22. Map labels, inspect image quality and group duplicates

Map source categories into the agreed 12-class taxonomy. Keep batteries separate from electronic waste. Exclude unrelated source labels and obvious figure/diagram filenames, decode RGB images, and reject images below 64 pixels in either dimension. Use file hashes and rotation/reflection-invariant perceptual hashes to group matching content. This automated filter does not replace a full manual label audit.

In [ ]:
import re
import hashlib
from PIL import Image, ImageOps, ImageDraw
from scipy.fft import dctn

LABELS_V3 = ["battery", "cardboard", "ceramics", "ewaste", "food_organics", "glass", "metal", "paper", "plastic", "textile_trash", "trash", "vegetation"]
MAP_HOUSE = {"battery":"battery", "biological":"food_organics", "brown-glass":"glass", "green-glass":"glass", "white-glass":"glass", "cardboard":"cardboard", "metal":"metal", "paper":"paper", "plastic":"plastic", "clothes":"textile_trash", "trash":"trash"}
MAP_EWASTE = {name:"ewaste" for name in ["pcb", "player", "microwave", "mobile", "mouse", "printer", "television", "washing machine", "washing_machine", "keyboard"]}
MAP_EWASTE["battery"] = "battery"
MAP_PHENOM = {"batteries":"battery", "e-waste":"ewaste", "ceramic_product":"ceramics", "ceramic_products":"ceramics", "coffee_tea_bags":"food_organics", "egg_shells":"food_organics", "food_scraps":"food_organics", "kitchen_waste":"food_organics", "yard_trimmings":"vegetation", "cans_all_type":"metal", "glass_containers":"glass", "paper_products":"paper", "plastic_bottles":"plastic", "stroform_product":"plastic", "styrofoam_product":"plastic"}

def canonical_phash(rgb):
    gray = rgb.convert("L").resize((32, 32))
    values = []
    for rotated in [gray, gray.transpose(Image.Transpose.ROTATE_90), gray.transpose(Image.Transpose.ROTATE_180), gray.transpose(Image.Transpose.ROTATE_270)]:
        for variant in [rotated, ImageOps.mirror(rotated)]:
            low = dctn(np.asarray(variant, dtype=float), norm="ortho")[:8, :8].ravel()[1:]
            bits = low > np.median(low)
            values.append(int.from_bytes(np.packbits(bits).tobytes(), "big"))
    return min(values)

candidates = clean_v2[["dataset", "label", "path"]].to_dict("records")
for source_name, mapping in [("household", MAP_HOUSE), ("ewaste", MAP_EWASTE), ("phenom", MAP_PHENOM)]:
    for path in sorted((RAW3 / source_name).rglob("*")):
        if path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}:
            continue
        label = mapping.get(path.parent.name.lower())
        if label is not None:
            candidates.append({"dataset": source_name, "label": label, "path": str(path)})

audit_rows = []
for index, row in enumerate(candidates):
    path = Path(row["path"])
    reason = "valid"
    if row["dataset"] == "phenom" and re.search(r"fig\d|fig_|diagram|chart|infograph|s2\.0|illustration|logo|icon|lifecycle|schematic", path.name, re.I):
        reason = "figure_or_diagram_filename"
    try:
        with Image.open(path) as original:
            rgb = ImageOps.exif_transpose(original).convert("RGB")
            rgb.load()
            if min(rgb.size) < 64:
                reason = "too_small"
            phash = canonical_phash(rgb)
            pixel_hash = hashlib.sha256(np.asarray(rgb.resize((128, 128))).tobytes()).hexdigest()
    except Exception as error:
        reason = "invalid_image"
        phash, pixel_hash = 0, ""
    audit_rows.append({**row, "sha256": hashlib.sha256(path.read_bytes()).hexdigest(), "pixel_hash": pixel_hash, "phash": phash, "status": reason})
    if (index + 1) % 5000 == 0:
        print("Audited images:", index + 1)

raw_inventory_v3 = pd.DataFrame(audit_rows)
raw_inventory_v3.to_csv(V3 / "raw_inventory.csv", index=False)
valid_v3 = raw_inventory_v3[raw_inventory_v3.status == "valid"].copy()
print("Mapped candidates:", len(raw_inventory_v3))
print("Decoded and accepted:", len(valid_v3))
display(pd.crosstab(valid_v3.label, valid_v3.dataset, margins=True))

sheet = Image.new("RGB", (8 * 150, 3 * 170), "white")
draw = ImageDraw.Draw(sheet)
for row_index, label in enumerate(["battery", "ewaste", "ceramics"]):
    examples = valid_v3[valid_v3.label == label].sample(n=min(8, len(valid_v3[valid_v3.label == label])), random_state=42)
    for column, (_, row) in enumerate(examples.iterrows()):
        with Image.open(row.path) as image:
            thumb = ImageOps.contain(image.convert("RGB"), (145, 140))
            sheet.paste(thumb, (column * 150, row_index * 170 + 22))
        draw.text((column * 150 + 2, row_index * 170 + 2), label, fill="black")
display(sheet)
sheet.save(V3 / "new_class_contact_sheet.jpg")

### 23. Build the final 20,000-image grouped split and class weights

Group identical files, matching resized pixels and conservatively close perceptual hashes before splitting. Exclude conflicting-label groups and retain one representative per group. Select exactly 20,000 distinct images by retaining scarce classes and applying the smallest feasible common cap to larger classes. Do not duplicate images or count augmentations toward this target. Stratify by source and class. Compute training-only weights as N_train / (12 * class_train_count), so aggregate weighted class contributions are equal. This does not guarantee unbiased predictions or unknown-object rejection.

In [ ]:
from scipy.spatial import cKDTree
from collections import defaultdict

valid_v3 = raw_inventory_v3[raw_inventory_v3.status == "valid"].copy()
ceramics_candidates3 = valid_v3[valid_v3.label == "ceramics"].sort_values("path").reset_index(drop=True)
assert len(ceramics_candidates3) == 134, "Source changed; repeat the ceramics visual review."
CERAMICS_APPROVED_INDEXES = [6,7,9,14,19,23,27,28,29,32,33,42,43,46,47,48,54,55,56,57,59,64,65,71,74,75,76,78,80,82,83,87,88,89,91,92,93,96,97,98,99,108,110,111,112,113,120,121,122,124,127,129,131,132]
ceramics_candidates3["manual_keep"] = ceramics_candidates3.index.isin(CERAMICS_APPROVED_INDEXES)
ceramics_candidates3.to_csv(V3 / "ceramics_manual_review.csv", index=False)
ceramics_allowed_paths3 = set(ceramics_candidates3[ceramics_candidates3.manual_keep].path)
valid_v3 = valid_v3[(valid_v3.label != "ceramics") | valid_v3.path.isin(ceramics_allowed_paths3)].reset_index(drop=True)
parent = list(range(len(valid_v3)))
def find_group(index):
    while parent[index] != index:
        parent[index] = parent[parent[index]]
        index = parent[index]
    return index

def merge_group(left, right):
    left, right = find_group(left), find_group(right)
    if left != right:
        parent[right] = left

for key in ("sha256", "pixel_hash"):
    for indexes in valid_v3.groupby(key).groups.values():
        indexes = list(indexes)
        for index in indexes[1:]:
            merge_group(indexes[0], index)
phash_values = valid_v3.phash.astype("uint64").to_numpy()
bits = ((phash_values[:, None] >> np.arange(64, dtype=np.uint64)) & 1).astype(float)
for left, right in cKDTree(bits).query_pairs(r=2, p=1):
    merge_group(left, right)
valid_v3["group_id"] = [find_group(i) for i in range(len(valid_v3))]
conflicts3 = set(valid_v3.groupby("group_id").label.nunique().loc[lambda values: values > 1].index)
conflicted3 = valid_v3[valid_v3.group_id.isin(conflicts3)]
conflicted3.to_csv(V3 / "conflicting_groups_excluded.csv", index=False)
unique3 = valid_v3[~valid_v3.group_id.isin(conflicts3)].drop_duplicates("group_id").copy()
counts_available3 = unique3.groupby("label").size().reindex(LABELS_V3)
TARGET_IMAGES3 = 20000
assert counts_available3.sum() >= TARGET_IMAGES3
low, high = 0, int(counts_available3.max())
while low < high:
    middle = (low + high) // 2
    if counts_available3.clip(upper=middle).sum() >= TARGET_IMAGES3:
        high = middle
    else:
        low = middle + 1
quotas3 = counts_available3.clip(upper=low).astype(int)
excess3 = int(quotas3.sum() - TARGET_IMAGES3)
for label in quotas3.sort_values(ascending=False).index[:excess3]:
    quotas3.loc[label] -= 1
assert quotas3.sum() == TARGET_IMAGES3
selected3 = pd.concat([
    group.sample(n=int(quotas3.loc[label]), random_state=42)
    for label, group in unique3.groupby("label", sort=True)
], ignore_index=True)
assert set(selected3.label) == set(LABELS_V3)
assert selected3.group_id.is_unique
rng3 = random.Random(42)
split_rows3 = []
for (dataset_name, label), group in selected3.groupby(["dataset", "label"], sort=True):
    records = group.sort_values("path").to_dict("records")
    rng3.shuffle(records)
    a = round(len(records) * 0.70)
    b = a + round(len(records) * 0.15)
    for split, records_part in [("train", records[:a]), ("validation", records[a:b]), ("test", records[b:])]:
        split_rows3.extend({**row, "split": split} for row in records_part)
manifest3 = pd.DataFrame(split_rows3)
manifest3.to_csv(V3 / "split_manifest.csv", index=False)
assert manifest3.groupby("group_id").split.nunique().max() == 1
assert len(manifest3) == 20000
training_counts3 = manifest3[manifest3.split == "train"].groupby("label").size().reindex(LABELS_V3)
assert training_counts3.notna().all() and (training_counts3 > 0).all()
class_weights3 = {i: float(training_counts3.sum() / (len(LABELS_V3) * count)) for i, count in enumerate(training_counts3)}
weighted_contributions3 = np.array([class_weights3[i] * count for i, count in enumerate(training_counts3)])
assert np.allclose(weighted_contributions3, weighted_contributions3[0])
weights_table3 = pd.DataFrame({"train_images": training_counts3, "class_weight": [class_weights3[i] for i in range(12)]})
weights_table3.to_csv(V3 / "class_weights.csv")
print("Valid candidate images:", len(valid_v3))
print("Unique non-conflicting groups:", len(unique3))
print("Selected distinct images:", len(manifest3))
print("Excluded conflicting groups:", len(conflicts3))
display(pd.crosstab(manifest3.label, manifest3.split, margins=True))
display(weights_table3)
print("Balanced contribution check passed; no detected duplicate group crosses a split.")

### 24. Extract ImageNet features for the new experiment

Start with fresh ImageNet weights rather than the old TrashNet-trained model, because the v3 split is new. Cache one feature vector per image so weighted and unweighted heads use identical inputs and can be compared efficiently on CPU. This stage uses frozen features and no online augmentation; later fine-tuning can add augmentation after the expanded baseline is established.

In [ ]:
tf.keras.utils.set_random_seed(42)
feature_backbone3 = tf.keras.applications.MobileNetV3Small(weights="imagenet", include_top=False, pooling="avg", input_shape=(224, 224, 3))
feature_backbone3.trainable = False
label_index3 = {label: index for index, label in enumerate(LABELS_V3)}
features3, targets3, ordered3 = {}, {}, {}

def load_rgb_array(encoded_path):
    if hasattr(encoded_path, "item"):
        encoded_path = encoded_path.item()
    with Image.open(encoded_path.decode("utf-8")) as image:
        return np.asarray(ImageOps.exif_transpose(image).convert("RGB"))

def decode_feature_image(path):
    image = tf.numpy_function(load_rgb_array, [path], tf.uint8)
    image.set_shape([None, None, 3])
    return tf.image.resize(tf.cast(image, tf.float32), (224, 224))

for split in ("train", "validation", "test"):
    frame = manifest3[manifest3.split == split].sort_values(["label", "path"]).reset_index(drop=True)
    ordered3[split] = frame
    data = tf.data.Dataset.from_tensor_slices(frame.path.to_list())
    data = data.map(decode_feature_image, num_parallel_calls=2).batch(32).prefetch(1)
    print("Extracting features:", split, len(frame))
    features3[split] = feature_backbone3.predict(data, verbose=1)
    targets3[split] = frame.label.map(label_index3).to_numpy(dtype=np.int32)
    assert len(features3[split]) == len(targets3[split]) and np.isfinite(features3[split]).all()
    np.savez_compressed(V3 / (split + "_features.npz"), features=features3[split], labels=targets3[split])
print("Feature extraction completed.")

### 25. Train matched unweighted and class-weighted models

Train two identical 12-class heads from the same initial weights. Only the weighted run receives class_weight. Both use the same split, seed, optimizer and maximum epochs. Keep validation and test distributions unchanged. Compare per-class recall and macro-F1 rather than claiming that weighted loss removes all bias.

In [ ]:
class ValidationMacroF1(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        predicted = self.model.predict(features3["validation"], verbose=0).argmax(axis=1)
        logs["val_macro_f1"] = f1_score(targets3["validation"], predicted, labels=np.arange(12), average="macro", zero_division=0)

heads3, histories3 = {}, {}
for experiment in ("unweighted", "weighted"):
    tf.keras.utils.set_random_seed(42)
    head = tf.keras.Sequential([
        tf.keras.Input(shape=(features3["train"].shape[1],)),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(len(LABELS_V3), activation="softmax"),
    ], name=experiment + "_head")
    head.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    print("Training:", experiment)
    histories3[experiment] = head.fit(
        features3["train"], targets3["train"],
        validation_data=(features3["validation"], targets3["validation"]),
        batch_size=32, epochs=25,
        class_weight=class_weights3 if experiment == "weighted" else None,
        callbacks=[ValidationMacroF1(), tf.keras.callbacks.EarlyStopping(monitor="val_macro_f1", mode="max", patience=4, restore_best_weights=True)],
        verbose=2,
    )
    heads3[experiment] = head
    head.save(V3 / (experiment + "_head.keras"))
    pd.DataFrame(histories3[experiment].history).to_csv(V3 / (experiment + "_training.csv"), index=False)

input3 = tf.keras.Input(shape=(224, 224, 3))
weighted_model3 = tf.keras.Model(input3, heads3["weighted"](feature_backbone3(input3, training=False)), name="EcoSort_balanced_12_classes")
weighted_model3.save(V3 / "ecosort_weighted_12class.keras")
(V3 / "class_names.json").write_text(json.dumps(LABELS_V3, indent=2))
print("Saved weighted 12-class model.")

### 26. Evaluate class balance and calibrate review scores

Evaluate both 12-class heads on the same held-out test images. Report macro-F1, balanced accuracy and per-class recall. Fit temperature scaling and a review threshold using validation data only. A review prompt is a confidence heuristic, not a guarantee of detecting unseen categories. Save predictions, reports, model settings and plots.

In [ ]:
from scipy.optimize import minimize_scalar
from scipy.special import softmax
from sklearn.metrics import balanced_accuracy_score, accuracy_score

val_probs3 = heads3["weighted"].predict(features3["validation"], verbose=0)
val_log3 = np.log(np.clip(val_probs3, 1e-8, 1))
val_counts3 = np.bincount(targets3["validation"], minlength=12)
val_weights3 = 1.0 / val_counts3[targets3["validation"]]
def temperature_loss(value):
    probs = softmax(val_log3 / value, axis=1)
    losses = -np.log(np.clip(probs[np.arange(len(probs)), targets3["validation"]], 1e-8, 1))
    return float(np.average(losses, weights=val_weights3))
temperature3 = float(minimize_scalar(temperature_loss, bounds=(0.5, 5.0), method="bounded").x)
val_calibrated3 = softmax(val_log3 / temperature3, axis=1)
val_scores3 = val_calibrated3.max(axis=1)
val_correct3 = val_calibrated3.argmax(axis=1) == targets3["validation"]
review_threshold3 = 0.75
threshold_found3 = False
for threshold in np.arange(0.50, 0.96, 0.025):
    accepted = val_scores3 >= threshold
    if accepted.mean() >= 0.20 and val_correct3[accepted].mean() >= 0.90:
        review_threshold3 = float(threshold)
        threshold_found3 = True
        break

comparison3_rows = []
for experiment, head in heads3.items():
    probabilities = head.predict(features3["test"], verbose=0)
    predicted = probabilities.argmax(axis=1)
    report = classification_report(targets3["test"], predicted, labels=np.arange(12), target_names=LABELS_V3, output_dict=True, zero_division=0)
    pd.DataFrame(report).transpose().to_csv(V3 / (experiment + "_classification_report.csv"))
    comparison3_rows.append({"model":experiment, "accuracy":accuracy_score(targets3["test"], predicted), "macro_f1":f1_score(targets3["test"], predicted, labels=np.arange(12), average="macro"), "balanced_accuracy":balanced_accuracy_score(targets3["test"], predicted)})
    predictions = ordered3["test"][["dataset", "label", "path"]].copy()
    predictions["predicted"] = [LABELS_V3[i] for i in predicted]
    predictions["confidence"] = probabilities.max(axis=1)
    predictions.to_csv(V3 / (experiment + "_test_predictions.csv"), index=False)
    fig, ax = plt.subplots(figsize=(10, 9))
    ConfusionMatrixDisplay(confusion_matrix(targets3["test"], predicted, labels=np.arange(12)), display_labels=LABELS_V3).plot(ax=ax, xticks_rotation=90, cmap="Blues", colorbar=False)
    fig.tight_layout()
    fig.savefig(V3 / (experiment + "_confusion_matrix.png"), dpi=160)
    plt.show()
comparison3 = pd.DataFrame(comparison3_rows).set_index("model")
comparison3.to_csv(V3 / "weighted_vs_unweighted.csv")
display(comparison3)
recall_comparison3 = pd.DataFrame({name:pd.read_csv(V3 / (name + "_classification_report.csv"), index_col=0).loc[LABELS_V3, "recall"] for name in heads3})
recall_comparison3.to_csv(V3 / "per_class_recall_comparison.csv")
display(recall_comparison3)
app_settings3 = {"classes":LABELS_V3, "temperature":temperature3, "review_threshold":review_threshold3, "validation_threshold_target_met":threshold_found3, "minimum_top_two_margin":0.10, "class_weights":class_weights3, "image_count":len(manifest3), "note":"Class-weighted loss balances training contribution; it does not ensure unbiased or reliable unknown-object recognition."}
(V3 / "app_settings.json").write_text(json.dumps(app_settings3, indent=2))
assert np.isfinite(temperature3) and 0 < review_threshold3 < 1
print("Temperature:", temperature3)
print("Review threshold:", review_threshold3)
print("Validation threshold target met:", threshold_found3)

### 27. Launch the balanced 12-class software

Use the weighted 12-class model, validation-derived score calibration and review threshold. Display Uncertain when confidence or the top-two margin is low. Keep all category scores visible for review. This prototype classifies an item category; it does not determine local disposal eligibility.

In [ ]:
DISPLAY3 = ["Battery", "Cardboard", "Ceramics", "Electronic waste", "Food organics", "Glass", "Metal", "Paper", "Plastic", "Textile waste", "Other waste", "Vegetation"]
def classify_balanced_waste(image):
    if image is None:
        raise gr.Error("Upload an image or take a photo first.")
    rgb = ImageOps.exif_transpose(image).convert("RGB")
    pixels = tf.image.resize(np.asarray(rgb, dtype=np.float32), (224, 224))
    raw_scores = weighted_model3(np.expand_dims(pixels.numpy(), 0), training=False).numpy()[0]
    scores = softmax(np.log(np.clip(raw_scores, 1e-8, 1)) / temperature3)
    if scores.shape != (12,) or not np.isfinite(scores).all():
        raise gr.Error("Invalid model output; please try another image.")
    order = np.argsort(scores)
    winner = int(order[-1])
    uncertain = float(scores[winner]) < review_threshold3 or float(scores[order[-1]] - scores[order[-2]]) < 0.10
    decision = "Uncertain / manual review" if uncertain else DISPLAY3[winner]
    message = f"{decision} | Top suggestion: {DISPLAY3[winner]} | score: {scores[winner]:.1%}"
    return dict(zip(DISPLAY3, map(float, scores))), message

sample3 = Path(ordered3["test"].iloc[0].path)
with Image.open(sample3) as sample:
    scores_check3, message_check3 = classify_balanced_waste(sample)
assert len(scores_check3) == 12 and np.isclose(sum(scores_check3.values()), 1.0)
print("12-class image pipeline check passed:", message_check3)
if "demo" in globals():
    demo.close()
demo3 = gr.Interface(
    fn=classify_balanced_waste,
    inputs=gr.Image(sources=["upload", "webcam"], type="pil", label="Waste item"),
    outputs=[gr.Label(num_top_classes=12, label="Category scores"), gr.Textbox(label="Decision", interactive=False)],
    examples=[[manifest3[manifest3.label == label].iloc[0].path] for label in LABELS_V3],
    title="EcoSort AI", flagging_mode="never",
)
demo3.launch(share=True, inline=True, debug=False)

### 28. Preserve the balanced-model evidence

Package the 12-class model, source metadata, split manifest, class weights, validation settings, test reports and figures. Include no raw third-party source images. Preserve the dataset references and licence descriptions for coursework attribution.

In [ ]:
RESULTS3 = V3 / "results_bundle"
RESULTS3.mkdir(exist_ok=True)
for path in V3.iterdir():
    if path.is_file() and path.suffix.lower() in {".csv", ".json", ".keras", ".png"}:
        shutil.copy2(path, RESULTS3 / path.name)
for path in RAW3.glob("*_metadata.json"):
    shutil.copy2(path, RESULTS3 / path.name)
(RESULTS3 / "README.txt").write_text("EcoSort 12-class research prototype.\nSource datasets: " + json.dumps(SOURCE_REFS) + "\nNo raw third-party images are redistributed.\nUse class_names.json and app_settings.json with ecosort_weighted_12class.keras.\n")
archive3 = shutil.make_archive(str(V3 / "EcoSort_balanced_12class_results"), "zip", RESULTS3)
print("Completed evidence archive:", archive3)
print("Final training images:", len(manifest3[manifest3.split == "train"]))
print("Final total images:", len(manifest3))
from google.colab import files
files.download(archive3)

### 29. Review the scarce ceramics source

Inspect ceramics candidates before final split creation. Exclude obvious off-topic images, diagrams, screenshots and collage-style instructional graphics. Record rejected source paths so the filter is reproducible.

In [ ]:
ceramics_review3 = valid_v3[valid_v3.label == "ceramics"].sort_values("path").reset_index(drop=True)
def show_ceramics_page(page):
    canvas = Image.new("RGB", (880, 432), "white")
    draw = ImageDraw.Draw(canvas)
    for offset in range(32):
        index = page * 32 + offset
        if index >= len(ceramics_review3):
            break
        path = ceramics_review3.iloc[index].path
        with Image.open(path) as image:
            thumb = ImageOps.contain(image.convert("RGB"), (106, 88))
            x, y = (offset % 8) * 110, (offset // 8) * 108
            canvas.paste(thumb, (x, y + 17))
            draw.text((x + 3, y + 1), str(index), fill="black")
    display(canvas)
print("Ceramics candidates:", len(ceramics_review3))
show_ceramics_page(4)

### 30. Final 20,000-image experiment summary

The final manifest contains **20,000 images across 12 classes**: 14,001 training, 3,004 validation and 2,995 test images. Exact and perceptual duplicate grouping was applied before splitting; this heuristic cannot guarantee detection of every related photograph. Manual ceramics review retained 54 images, including only eight test images.

This expanded experiment uses a fresh frozen ImageNet MobileNetV3Small backbone and matched classification heads, not completed 12-class backbone fine-tuning. Training-only class weights are N_train / (12 * class_count). Both heads are selected by validation macro-F1.

| Test metric | Unweighted | Class-weighted |
| --- | ---: | ---: |
| Accuracy | 85.48% | 84.17% |
| Macro-F1 | 0.8071 | 0.7968 |
| Mean class recall | 79.62% | 81.61% |

Weighting improved mean class recall, not every metric. Ceramics recall rose from 25% to 50%, based on eight test images and therefore highly uncertain. Battery recall rose from 88.46% to 92.95%; vegetation recall rose from 87.06% to 94.12%. Some larger classes lost recall.

The application uses validation-only temperature scaling (T = 1.0608), a 0.55 review threshold and a 0.10 minimum top-two margin. These heuristics do not guarantee unfamiliar-object detection. The software predicts item categories, not local disposal eligibility. Textile photographs may depict ordinary clothing, not confirmed waste.

Evidence archive: /content/EcoSort_dataset_v3/EcoSort_balanced_12class_results.zip. Colab /content storage is temporary. The archive preserves the model and reports but does not redistribute raw third-party images. Webcam capture has not been tested.
